<h1 style="color: #17638c; font-size: 24px; font-weight: 750; margin: 2px 0 16px 0; line-height: 1.35;">Data Collection</h1><div style="background-color: #eef6fc; border-left: 4px solid #3498db; padding: 12px 16px; border-radius: 5px; color: #243746; font-size: 13.5px; line-height: 1.65; margin: 8px 0 17px 0;"><p style="margin: 3px 0 10px 0;">We use a different approach for each website:</p><table style="border-collapse: collapse; width: 100%; font-size: 13px; margin: 10px 0 12px 0;">
<thead>
<tr>
<th style="background: #d9eaf6; text-align: left; padding: 9px 10px; border: 1px solid #c3d8e9; color: #1e526f;">Website</th>
<th style="background: #d9eaf6; text-align: left; padding: 9px 10px; border: 1px solid #c3d8e9; color: #1e526f;">Collection method</th>
<th style="background: #d9eaf6; text-align: left; padding: 9px 10px; border: 1px solid #c3d8e9; color: #1e526f;">Why</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;"><strong>Bien’ici</strong></td>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;"><code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">requests</code> + JSON API</td>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;">Listing information is returned as structured JSON.</td>
</tr>
<tr>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;"><strong>PAP</strong></td>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;">Playwright + BeautifulSoup + targeted regex</td>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;">Listing pages are opened in a browser; Jupyter on Windows runs the browser code in a separate Python process.</td>
</tr>
<tr>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;"><strong>ParuVendu</strong></td>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;"><code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">requests</code> + BeautifulSoup</td>
<td style="padding: 9px 10px; border: 1px solid #c3d8e9; vertical-align: top;">We inspected the HTML labels and use them to read the correct values, without regex.</td>
</tr>
</tbody>
</table></div>


<h2 style="color: #2471a3; font-size: 21px; font-weight: 700; margin: 21px 0 11px 0; line-height: 1.35;">0. Environment and output files</h2><div style="background-color: #eef6fc; border-left: 4px solid #3498db; padding: 12px 16px; border-radius: 5px; color: #243746; font-size: 13.5px; line-height: 1.65; margin: 8px 0 17px 0;"><p style="margin: 3px 0 10px 0;">Run these cells in order. All live collection switches are <strong>off by default</strong> so opening the notebook does not immediately make many website requests. We recommend checking the test cells first, then enabling one source at a time.</p><p style="margin: 3px 0 10px 0;">If required, install packages in the <strong>same Python environment used by this notebook</strong>:</p><pre style="background: #f8fbfe; border: 1px solid #d1e5f2; padding: 10px 12px; border-radius: 5px; font-size: 12.5px; line-height: 1.5; overflow-x: auto;"><code class="language-python">%pip install requests beautifulsoup4 pandas playwright
</code></pre><p style="margin: 3px 0 10px 0;">For PAP, Playwright also needs its Chromium browser; this is a separate installation step:</p><pre style="background: #f8fbfe; border: 1px solid #d1e5f2; padding: 10px 12px; border-radius: 5px; font-size: 12.5px; line-height: 1.5; overflow-x: auto;"><code class="language-python">import subprocess, sys
subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], check=True)
</code></pre></div>


In [1]:
from pathlib import Path
import json
import time
import re
import sys
import os
import subprocess
import unicodedata
from urllib.parse import urljoin, urlparse

import requests
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display

OUTPUT_DIR = Path('marseille_scraping_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Output folder:', OUTPUT_DIR.resolve())
print('Python interpreter:', sys.executable)

Output folder: C:\Users\romay\OneDrive\BOUKRIF_Romayssa_CHAHINE_Rami_G5\BOUKRIF_Romayssa_CHAHINE_Rami_G5_Code\marseille_scraping_outputs
Python interpreter: C:\Users\romay\anaconda3\python.exe


<h1 style="color: #17638c; font-size: 24px; font-weight: 750; margin: 2px 0 16px 0; line-height: 1.35;">1. Bien’ici: use the JSON API</h1><div style="background-color: #eef6fc; border-left: 4px solid #3498db; padding: 12px 16px; border-radius: 5px; color: #243746; font-size: 13.5px; line-height: 1.65; margin: 8px 0 17px 0;"><p style="margin: 3px 0 10px 0;">The first API call looks up Marseille geographic IDs. The second one requests rental listings in batches. The API may change its fields, paging rules, or availability.</p></div>


In [2]:
BIENICI_HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Referer': 'https://www.bienici.com/',
}


def bi_get_marseille_zones(session):
    """Ask the suggestion API for geographic identifiers for Marseille."""
    zones = []
    queries = ['Marseille'] + [f'Marseille 130{i:02d}' for i in range(1, 17)]
    for query in queries:
        try:
            response = session.get(
                'https://res.bienici.com/suggest.json',
                params={'q': query}, timeout=10,
            )
            response.raise_for_status()
            for item in response.json():
                postal = str(item.get('postalCode') or '')
                city = str(item.get('name') or '').lower()
                if postal.startswith('130') or 'marseille' in city:
                    for zone_id in (item.get('zoneIds') or []):
                        if zone_id not in zones:
                            zones.append(zone_id)
                    break
        except (requests.RequestException, ValueError, TypeError) as error:
            print(f'Zone lookup failed for {query}: {error}')
        time.sleep(0.2)
    print(f'Found {len(zones)} Bien’ici zone IDs.')
    return zones


def bi_fetch_raw_ads(max_results=1000):
    raw_ads = []
    with requests.Session() as session:
        session.headers.update(BIENICI_HEADERS)
        zones = bi_get_marseille_zones(session)
        if not zones:
            return raw_ads
        for offset in range(0, max_results, 100):
            filters = {
                'size': min(100, max_results - offset),
                'from': offset,
                'filterType': 'rent',
                'propertyType': ['house', 'flat', 'loft'],
                'zoneIdsByTypes': {'zoneIds': zones},
                'onTheMarket': [True],
            }
            try:
                response = session.get(
                    'https://www.bienici.com/realEstateAds.json',
                    params={'filters': json.dumps(filters)}, timeout=15,
                )
                response.raise_for_status()
                ads = response.json().get('realEstateAds', [])
            except (requests.RequestException, ValueError, TypeError) as error:
                print(f'API request stopped at offset {offset}: {error}')
                break
            if not ads:
                print('The API returned no more listings.')
                break
            raw_ads.extend(ads)
            print(f'Offset {offset}: received {len(ads)} listings')
            time.sleep(1.5)
    return raw_ads


def bi_optional_boolean(value):
    """Avoid interpreting a missing JSON field as an explicit No."""
    if value is True:
        return 'Oui'
    if value is False:
        return 'Non'
    return None


def bi_flatten_ad(ad):
    """Choose useful fields from one API dictionary"""
    description_html = ad.get('description') or ''
    description = BeautifulSoup(description_html, 'html.parser').get_text(' ', strip=True) or None
    blur = ad.get('blurInfo') or {}
    position = blur.get('position') or blur.get('centroid') or {}
    district = ad.get('district') or {}
    balcony = ad.get('hasBalcony')
    terrace = ad.get('hasTerrace')
    outdoor = (True if balcony is True or terrace is True
               else False if balcony is False and terrace is False else None)
    postal = ad.get('postalCode')
    ad_id = ad.get('id')

    return {
        'source_site': 'Bienici',
        'id': ad_id,
        'url': ad.get('url'),  # Do not guess an undocumented public listing URL.
        'title': ad.get('title'),
        'property_type': ad.get('propertyType'),
        'price_total_eur': ad.get('price'),
        'charges_eur': ad.get('charges'),
        'depot_garantie_eur': ad.get('safetyDeposit'),
        'honoraires_eur': ad.get('agencyRentalFee'),
        'surface_m2': ad.get('surfaceArea'),
        'rooms': ad.get('roomsQuantity'),
        'bedrooms': ad.get('bedroomsQuantity'),
        'bathrooms': ad.get('bathroomsQuantity'),
        'floor': ad.get('floor'),
        'city': ad.get('city'),
        'zipcode': str(postal).strip() if postal is not None else None,
        'district': district.get('name'),
        'latitude': position.get('lat') if position.get('lat') is not None else ad.get('lat'),
        'longitude': position.get('lon') if position.get('lon') is not None else ad.get('lon'),
        'dpe_class': ad.get('energyClassification'),
        'ges_class': ad.get('greenhouseGazClassification'),
        'heating': ad.get('heating'),
        'is_furnished': bi_optional_boolean(ad.get('isFurnished')),
        'has_elevator': bi_optional_boolean(ad.get('hasElevator')),
        'has_balcony_terrace': bi_optional_boolean(outdoor),
        'has_cellar': bi_optional_boolean(ad.get('hasCellar')),
        'has_parking': bi_optional_boolean(ad.get('hasParking')),
        # Publication dates are deliberately excluded from the structured CSV.
        'raw_text': description,
    }


def bi_flatten_ads(raw_ads):
    """Create one row per listing ID """
    seen_ids = set()
    rows = []
    for ad in raw_ads:
        ad_id = ad.get('id')
        if ad_id is not None:
            if ad_id in seen_ids:
                continue
            seen_ids.add(ad_id)
        rows.append(bi_flatten_ad(ad))
    return pd.DataFrame(rows)    

In [3]:
RUN_BIENICI = True  # Set to True only when you want to query the live API.
BIENICI_MAX_RESULTS = 1000  

if RUN_BIENICI:
    bi_raw_ads = bi_fetch_raw_ads(max_results=BIENICI_MAX_RESULTS)
    df_bienici = bi_flatten_ads(bi_raw_ads)
    if not df_bienici.empty:
        df_bienici.to_csv(OUTPUT_DIR / 'bienici.csv', index=False, encoding='utf-8-sig')
        print('Saved', len(df_bienici), 'structured Bien’ici rows.')
        display(df_bienici.head())
    else:
        print('No Bien’ici rows were collected.')
else:
    print('Bien’ici collection is off. Enable RUN_BIENICI to make live requests.')

Found 17 Bien’ici zone IDs.
Offset 0: received 100 listings
Offset 100: received 100 listings
Offset 200: received 100 listings
Offset 300: received 100 listings
Offset 400: received 100 listings
Offset 500: received 100 listings
Offset 600: received 85 listings
Offset 700: received 85 listings
Offset 800: received 85 listings
Offset 900: received 85 listings
Saved 685 structured Bien’ici rows.


,source_site,id,url,title,property_type,price_total_eur,charges_eur,depot_garantie_eur,honoraires_eur,surface_m2,...,longitude,dpe_class,ges_class,heating,is_furnished,has_elevator,has_balcony_terrace,has_cellar,has_parking,raw_text
0,Bienici,ag132391-555657911,None,Atelier / Local / Stockage de 17m2 Quartier Ti...,loft,285.00,35.00,NaN,350.00,17.00,...,5.397644,NS,NS,NaN,NaN,NaN,NaN,NaN,None,Atelier / Local / Stockage de 17m2 Quartier Ti...
1,Bienici,hektor-xavierradisson13-203,None,Studio 26 rue Aramis 13014,flat,359.00,69.00,290.0,187.00,14.26,...,5.386046,C,B,électrique individuel,Non,NaN,NaN,NaN,None,Contact : 06 80 12 05 66 . Situé 26 rue Aramis...
2,Bienici,seiitra-elyade_bi-17290,None,,flat,392.00,65.00,327.0,246.91,18.82,...,5.432999,A,A,Convecteurs Electrique Individuel,Non,Oui,NaN,NaN,None,Résidence étudiante idéalement située à Marsei...
3,Bienici,ag135528-554266103,None,T1,flat,407.92,43.92,NaN,177.39,13.00,...,5.384386,NS,NS,collectif,Non,NaN,NaN,NaN,None,"Découvrez cette chambre de bonne de 13,52 m², ..."
4,Bienici,ag752345-556792673,None,La Blancarde - 4 chambres,flat,410.00,31.00,758.0,276.00,10.00,...,5.401141,C,D,gaz collectif,Oui,Oui,NaN,NaN,None,Logement Manda ! Visitez et déposez votre doss...


<h1 style="color: #17638c; font-size: 24px; font-weight: 750; margin: 2px 0 16px 0; line-height: 1.35;">2.  ParuVendu: web scraping</h1><div style="background-color: #eef6fc; border-left: 4px solid #3498db; padding: 12px 16px; border-radius: 5px; color: #243746; font-size: 13.5px; line-height: 1.65; margin: 8px 0 17px 0;"><p style="margin: 3px 0 10px 0;">For ParuVendu, we use the methods covered in class: <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">find()</code>, <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">find_all()</code>, <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">select()</code> and <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">get_text()</code>. We inspected individual listing HTML and found where the advertised rent and the financial details are displayed.</p><ul style="padding-left: 21px; margin: 8px 0 9px 0;">
<li style="margin: 4px 0;">Rent: <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">&lt;span class="text-4.5xl"&gt;730 € CC&lt;/span&gt;</code> (charges already included).</li>
<li style="margin: 4px 0;">Charges, deposit and fees: <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">&lt;div class="opt19_hd_det"&gt;</code> with a label in <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">&lt;span&gt;</code> and a value in <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">&lt;strong&gt;</code>.</li>
<li style="margin: 4px 0;">Rooms and surface: labelled <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">&lt;li&gt;</code> elements.</li>
<li style="margin: 4px 0;">Postal code: explicitly given in the ad title, for example <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">(13006)</code>; <strong>not assumed from the search URL</strong> (a search may include nearby towns).</li>
<li style="margin: 4px 0;">DPE: the letter is the <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">NoteActive</code> span; consumption comes from <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">DPE_consEnerTxt</code>, and greenhouse-gas emissions from <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">DPE_effSerreTxt</code>.</li>
</ul><p style="margin: 3px 0 10px 0;">A <strong>GES emissions number</strong> is not the same as a <strong>GES letter</strong>. If no GES class is provided, we keep <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">ges_class</code> missing. Likewise, an absent feature does not automatically mean <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">Non</code>.</p></div>


In [4]:

import re
import unicodedata


def normalized(text):
    """Lowercase, remove accents, and normalize spaces for label comparisons."""
    decomposed = unicodedata.normalize("NFKD", str(text).casefold())
    no_accents = "".join(
        ch for ch in decomposed if not unicodedata.combining(ch)
    )
    return " ".join(no_accents.split())


def to_euros(text):
    """Convert '1 250,50 €' or '730 € CC' to float, without regex."""
    if not text or "€" not in text:
        return None

    amount = text.split("€", 1)[0].strip()
    amount = amount.replace(" ", "").replace("\u00a0", "").replace("\u202f", "")
    amount = amount.replace(".", "").replace(",", ".")

    try:
        return float(amount)
    except ValueError:
        return None


def to_number(text):
    """Read numbers like '75m2 environ', '2 pièces', '1 chambre'."""
    if not text:
        return None

    value = normalized(text).replace(",", ".")

    # Only parse the leading number, not numbers later in a sentence.
    token = []

    for char in value:
        if char.isdigit() or (char == "." and "." not in token):
            token.append(char)
        else:
            break

    if not token:
        return None

    try:
        return float("".join(token))
    except ValueError:
        return None


def get_characteristic_text(soup, name):
    """Read a value inside a <li> carrying the requested label."""
    expected = normalized(name)

    for li in soup.find_all("li"):
        label = li.find("span", class_="uppercase")

        if label is not None and normalized(
            label.get_text(" ", strip=True)
        ) == expected:

            value = li.find("span", class_="text-lg")

            if value is not None:
                return value.get_text(" ", strip=True)

    return None


def get_yes_no(soup, label):
    """Only return Yes/No if the labelled HTML explicitly states it."""
    value = get_characteristic_text(soup, label)

    if value is None:
        return None

    value = normalized(value)

    if value in ("oui", "yes"):
        return "Oui"

    if value in ("non", "no"):
        return "Non"

    return None


def get_energy_grade(soup, label):
    """Return a letter A-G only when a labelled property field provides it."""
    value = get_characteristic_text(soup, label)

    if not value:
        return None

    value = value.strip().upper()

    return value if len(value) == 1 and value in "ABCDEFG" else None


def postal_code_in_text(text):
    """Return a five-digit postal code explicitly written in the given text."""
    if not text:
        return None

    for word in str(text).replace("(", " ").replace(")", " ").split():
        candidate = word.strip(" ,;:.!?[]{}")

        if len(candidate) == 5 and candidate.isdigit() and candidate != "00000":
            return candidate

    return None


def get_postal_code(soup):
    """Read the postal code from the ad itself, not the search URL."""

    # First check whether the page explicitly labels the postcode.
    labelled_value = get_characteristic_text(soup, "Code postal")
    postcode = postal_code_in_text(labelled_value)

    if postcode is not None:
        return postcode

    # The postal code may also appear in the page title.
    title_text = soup.title.get_text(" ", strip=True) if soup.title else None

    return postal_code_in_text(title_text)


def extract_listing_details(soup):
    """Extract property information from the advertisement HTML."""

    record = {
        "price_total_eur": None,
        "charges_eur": None,
        "depot_garantie_eur": None,
        "honoraires_eur": None,
        "surface_m2": None,
        "rooms": None,
        "bedrooms": None,
        "zipcode": None,
        "dpe_class": None,
        "ges_class": None,
        "dpe_consumption_kwh_m2_an": None,
        "ges_emissions_kg_co2_m2_an": None,
        "is_furnished": None,
        "has_elevator": None,
        "has_balcony_terrace": None,
        "heating": None,
        "floor": None,
        "exposure": None,
    }

    # 1. RENT
    # Select the main price span instead of the first amount on the page.
    for span in soup.find_all("span", class_="text-4.5xl"):
        text = span.get_text(" ", strip=True)

        if "€" in text and "CC" in text.upper().split():
            record["price_total_eur"] = to_euros(text)
            break

    # 2. FINANCIAL DETAILS
    # Each block contains a label in <span> and an amount in <strong>.
    for detail in soup.find_all("div", class_="opt19_hd_det"):
        label_span = detail.find("span")
        amount_strong = detail.find("strong")

        if label_span is None or amount_strong is None:
            continue

        label = normalized(label_span.get_text(" ", strip=True))
        amount = to_euros(amount_strong.get_text(" ", strip=True))

        if "charges/mois" in label or "charges / mois" in label:
            record["charges_eur"] = amount

        elif "depot" in label and "garantie" in label:
            record["depot_garantie_eur"] = amount

        elif "honoraires" in label:
            record["honoraires_eur"] = amount

    # 3. SURFACE AREA
    surface_text = get_characteristic_text(soup, "Surface")

    if surface_text is None:
        # Some pages display the surface without a specific label.
        for span in soup.find_all("span", class_="text-lg"):
            text = normalized(span.get_text(" ", strip=True))

            if text and text[0].isdigit() and ("m2" in text or "m²" in text):
                surface_text = text
                break

    record["surface_m2"] = to_number(surface_text)

    # 4. NUMBER OF ROOMS AND BEDROOMS
    # Rooms and bedrooms are two different characteristics.
    room_count = to_number(
        get_characteristic_text(soup, "Nombre de pièces")
    )

    bedroom_count = to_number(
        get_characteristic_text(soup, "Nombre de chambres")
    )

    record["rooms"] = int(room_count) if room_count is not None else None
    record["bedrooms"] = int(bedroom_count) if bedroom_count is not None else None

    # 5. GENERAL INFORMATION
    # Extract heating, floor, and exposure from labelled HTML values.
    for li in soup.find_all("li"):
        label = li.find("span", class_="uppercase")

        if label is None or normalized(
            label.get_text(" ", strip=True)
        ) != "general":
            continue

        for value_span in li.find_all("span", class_="text-lg"):
            item = value_span.get_text(" ", strip=True)

            if ":" not in item:
                continue

            key, value = item.split(":", 1)
            key = normalized(key)
            value = value.strip()

            if key == "chauffage":
                record["heating"] = value

            elif key == "etage":
                record["floor"] = value

            elif key == "exposition":
                record["exposure"] = value

    # 6. POSTAL CODE
    record["zipcode"] = get_postal_code(soup)

    # 7. ENERGY PERFORMANCE
    # Identify the active DPE grade from the energy scale.
    dpe_scales = soup.find_all("div", class_="DPE_ng_flex")

    if len(dpe_scales) == 1:
        active_grade = dpe_scales[0].find("span", class_="NoteActive")

        if active_grade is not None:
            letter = active_grade.get_text(" ", strip=True).upper()

            if len(letter) == 1 and letter in "ABCDEFG":
                record["dpe_class"] = letter

    if record["dpe_class"] is None:
        record["dpe_class"] = get_energy_grade(soup, "DPE")

    # Energy consumption in kWh/m²/year.
    consumption = soup.find("div", class_="DPE_consEnerTxt")

    if consumption is not None:
        consumption_span = consumption.find("span")

        if consumption_span is not None:
            record["dpe_consumption_kwh_m2_an"] = to_number(
                consumption_span.get_text(" ", strip=True)
            )

    # Greenhouse gas emissions in kgCO2/m²/year.
    emissions = soup.find("div", class_="DPE_effSerreTxt")

    if emissions is not None:
        emissions_span = emissions.find("span")

        if emissions_span is not None:
            record["ges_emissions_kg_co2_m2_an"] = to_number(
                emissions_span.get_text(" ", strip=True)
            )

    # Only extract the GES letter if it is explicitly provided.
    record["ges_class"] = get_energy_grade(soup, "GES")

    # 8. FURNISHED STATUS — EXTRACTED FROM THE DESCRIPTION
    # ParuVendu does not always provide a dedicated furnished field.
    # We therefore analyze the advertisement description using regex.
    description_tag = soup.find("div", id="txtAnnonceTrunc")

    if description_tag is not None:
        description = normalized(
            description_tag.get_text(" ", strip=True)
        )


        negative_pattern = (
            r"\b(?:non[\s-]+meuble(?:e|s|es)?|"
            r"pas[\s-]+meuble(?:e|s|es)?|"
            r"sans[\s-]+meubles?|"
            r"location[s]?\s+nue[s]?)\b"
        )

        positive_pattern = r"\bmeuble(?:e|s|es)?\b"

        negative_match = re.search(negative_pattern, description)
        positive_matches = list(re.finditer(positive_pattern, description))

        # Ignore furnished mentions that belong to a negative expression.
        positive_found = any(
            not (
                negative_match is not None
                and negative_match.start() <= match.start()
                and match.end() <= negative_match.end()
            )
            for match in positive_matches
        )

        if negative_match and positive_found:
            # Conflicting statements: do not guess.
            record["is_furnished"] = None

        elif negative_match:
            record["is_furnished"] = "Non"

        elif positive_found:
            record["is_furnished"] = "Oui"

    # 9. OTHER OPTIONAL FEATURES
    record["has_elevator"] = get_yes_no(soup, "Ascenseur")
    record["has_balcony_terrace"] = get_yes_no(soup, "Balcon / Terrasse")

    return record


In [5]:
PV_MAX_ARRONDISSEMENTS = 16
PV_PAGES_PER_ARRONDISSEMENT = 5
PV_MAX_ADS = None
RUN_PARUVENDU = True

PV_BASE = 'https://www.paruvendu.fr'
pv_session = requests.Session()
pv_session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7',
})


def pv_fetch_soup(url):
    """Download one page, or return None if access fails."""
    try:
        response = pv_session.get(url, timeout=15)
        response.raise_for_status()
    except requests.RequestException as error:
        print(f'Could not download {url}: {error}')
        return None
    soup = BeautifulSoup(response.text, 'html.parser')
    page_title = soup.title.get_text(' ', strip=True).lower() if soup.title else ''
    if any(text in page_title for text in ('just a moment', 'cloudflare', 'access denied')):
        print(f'Access challenge at {url}')
        return None
    return soup


def pv_collect_urls(max_arrondissements, pages_per_arrondissement):
    """Collect unique rental-apartment and rental-house URLs by arrondissement."""
    unique = set()
    for arrondissement in range(1, max_arrondissements + 1):
        search_zip = f'130{arrondissement:02d}'
        for page_number in range(1, pages_per_arrondissement + 1):
            search_url = (
                'https://www.paruvendu.fr/immobilier/recherche/location/'
                f'appartement/marseille-{search_zip}/?p={page_number}'
            )
            print(f'ParuVendu: postcode {search_zip}, page {page_number}')
            soup = pv_fetch_soup(search_url)
            if soup is None:
                break
            before = len(unique)
            for tag in soup.find_all('a', href=True):
                candidate = urljoin(PV_BASE, tag['href'])
                parsed = urlparse(candidate)
                if parsed.netloc not in ('www.paruvendu.fr', 'paruvendu.fr'):
                    continue
                if parsed.path.startswith((
                    '/immobilier/location/appartement/',
                    '/immobilier/location/maison/',
                )):
                    unique.add(candidate.split('?', 1)[0].split('#', 1)[0])
            print('  New listing URLs:', len(unique) - before)
            if len(unique) == before:
                break
            time.sleep(1)
    return sorted(unique)


def pv_scrape(max_arrondissements=2, pages_per_arrondissement=1, max_ads=10):
    """Open every selected ad and collect the HTML fields into dictionaries."""
    urls = pv_collect_urls(max_arrondissements, pages_per_arrondissement)
    print('Unique listing links:', len(urls))
    selected = urls if max_ads is None else urls[:max_ads]
    records = []
    for index, url in enumerate(selected, start=1):
        print(f'ParuVendu listing {index}/{len(selected)}')
        soup = pv_fetch_soup(url)
        if soup is None:
            continue
        record = extract_listing_details(soup)
        record['source_site'] = 'ParuVendu'
        record['url'] = url
        record['property_type'] = 'maison' if '/location/maison/' in url else 'appartement'
        record['raw_text'] = soup.get_text(' ', strip=True)[:2000]
        records.append(record)
        time.sleep(0.75)
    return pd.DataFrame(records)

In [6]:



if RUN_PARUVENDU:
    df_paruvendu = pv_scrape(
        max_arrondissements=PV_MAX_ARRONDISSEMENTS,
        pages_per_arrondissement=PV_PAGES_PER_ARRONDISSEMENT,
        max_ads=PV_MAX_ADS,
    )
    if not df_paruvendu.empty:
        df_paruvendu.to_csv(
            OUTPUT_DIR / 'paruvendu.csv',
            index=False, encoding='utf-8-sig',
        )
        print('Saved', len(df_paruvendu), 'ParuVendu records.')
        display(df_paruvendu.head())
    else:
        print('No ParuVendu rows were collected.')
else:
    print('ParuVendu collection is off.')

ParuVendu: postcode 13001, page 1
  New listing URLs: 25
ParuVendu: postcode 13001, page 2
Could not download https://www.paruvendu.fr/immobilier/recherche/location/appartement/marseille-13001/?p=2: 404 Client Error: Not found for url: https://www.paruvendu.fr/immobilier/recherche/location/appartement/marseille-13001/?p=2
ParuVendu: postcode 13002, page 1
  New listing URLs: 10
ParuVendu: postcode 13002, page 2
Could not download https://www.paruvendu.fr/immobilier/recherche/location/appartement/marseille-13002/?p=2: 404 Client Error: Not found for url: https://www.paruvendu.fr/immobilier/recherche/location/appartement/marseille-13002/?p=2
ParuVendu: postcode 13003, page 1
  New listing URLs: 7
ParuVendu: postcode 13003, page 2
Could not download https://www.paruvendu.fr/immobilier/recherche/location/appartement/marseille-13003/?p=2: 404 Client Error: Not found for url: https://www.paruvendu.fr/immobilier/recherche/location/appartement/marseille-13003/?p=2
ParuVendu: postcode 13004, pa

,price_total_eur,charges_eur,depot_garantie_eur,honoraires_eur,surface_m2,rooms,bedrooms,zipcode,dpe_class,ges_class,...,is_furnished,has_elevator,has_balcony_terrace,heating,floor,exposure,source_site,url,property_type,raw_text
0,1150.0,NaN,NaN,NaN,75.0,3,NaN,NaN,NaN,None,...,NaN,None,None,NaN,NaN,NaN,ParuVendu,https://www.paruvendu.fr/immobilier/location/a...,appartement,Annonce tel beau t3 impeccable quartier calme ...
1,1490.0,NaN,NaN,NaN,40.0,2,NaN,13006,NaN,None,...,Oui,None,None,NaN,NaN,NaN,ParuVendu,https://www.paruvendu.fr/immobilier/location/a...,appartement,Annonce location appartement 2 pièces de 40m2 ...
2,555.0,NaN,NaN,NaN,35.0,2,NaN,13014,E,None,...,NaN,None,None,NaN,NaN,NaN,ParuVendu,https://www.paruvendu.fr/immobilier/location/a...,appartement,Annonce location appartement 2 pièces de 35m2 ...
3,1490.0,NaN,NaN,NaN,46.0,2,NaN,13006,NaN,None,...,Oui,None,None,NaN,NaN,NaN,ParuVendu,https://www.paruvendu.fr/immobilier/location/a...,appartement,Annonce location appartement 2 pièces de 46m2 ...
4,2650.0,333.0,4634.0,1048.0,81.0,3,2.0,13008,NaN,None,...,Oui,None,None,NaN,2,NaN,ParuVendu,https://www.paruvendu.fr/immobilier/location/a...,appartement,Annonce location appartement 3 pièces de 81m2 ...


<h1 style="color: #17638c; font-size: 24px; font-weight: 750; margin: 2px 0 16px 0; line-height: 1.35;">3. PAP (scraping  with Playwright)</h1><div style="background-color: #eef6fc; border-left: 4px solid #3498db; padding: 12px 16px; border-radius: 5px; color: #243746; font-size: 13.5px; line-height: 1.65; margin: 8px 0 17px 0;"><h3 style="color: #2471a3; font-size: 16px; font-weight: 700; margin: 13px 0 6px 0;">Why do we use Playwright instead of requests?</h3><p style="margin: 3px 0 10px 0;">We first tried using <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">requests</code> and <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">BeautifulSoup</code>, as we did for the other websites. However, PAP returned an HTTP 403 error and displayed a Cloudflare verification page instead of the property listings.</p><p style="margin: 3px 0 10px 0;">We therefore decided to use <strong>Playwright</strong>, which allows Python to control a real Chromium browser. It can open web pages, execute JavaScript, interact with elements such as cookie buttons, and retrieve the HTML once the page has loaded.</p><p style="margin: 3px 0 10px 0;">We still use <strong>BeautifulSoup</strong> to parse the HTML and extract the information we need, such as property titles, prices, surfaces and other characteristics.</p><p style="margin: 3px 0 10px 0;">Playwright does not guarantee access when Cloudflare blocks automated browsing, but it gives us a more suitable way to handle browser-based pages.</p><h3 style="color: #2471a3; font-size: 16px; font-weight: 700; margin: 13px 0 6px 0;">Why do we use a separate <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">.py</code> file?</h3><p style="margin: 3px 0 10px 0;">We initially tried running Playwright directly inside our Jupyter notebook, but we encountered compatibility errors related to the <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">asyncio</code> event loop in our Windows/Anaconda environment.</p><p style="margin: 3px 0 10px 0;">To avoid this issue, we use <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">%%writefile</code> to save the scraping code as a Python script called <code style="background: #e4edf5; color: #244766; padding: 1px 4px; border-radius: 3px; font-size: 0.95em;">pap_worker.py</code>.</p><p style="margin: 3px 0 10px 0;">We then execute this script in a separate Python process from Jupyter. This allows Playwright to open Chromium without conflicting with Jupyter's event loop.</p></div>


In [7]:
%%writefile pap_worker.py
"""Collect PAP rental advertisements for the Marseille housing project.

This file is generated directly from the notebook. It runs separately because
sync_playwright() conflicts with the asyncio loop used by Jupyter on Windows.

The code keeps the visible Chromium setup from the version that worked locally.
Only residential listings are saved to the dataset.
"""

import argparse
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup
from playwright.sync_api import sync_playwright
from playwright.sync_api import TimeoutError as PlaywrightTimeoutError
from playwright._impl._errors import TargetClosedError

BASE_URL = "https://www.pap.fr/annonce/locations-marseille-13-g12024"
PAP_DOMAIN = "https://www.pap.fr"

# These words describe the property category in an ad title.
# We check the title, not the full description: an apartment may contain a desk.
RESIDENTIAL_TYPES = {
    "appartement": "appartement", "studio": "studio",
    "maison": "maison", "villa": "maison",
    "loft": "loft", "duplex": "duplex",
    "chambre": "chambre", "colocation": "colocation",
}
NON_RESIDENTIAL_TYPES = {
    "bureau", "bureaux", "commerce", "commerces", "commercial",
    "local", "locaux", "boutique", "boutiques", "parking",
    "garage", "garages", "entrepôt", "entrepot", "entrepôts",
    "entrepots", "terrain", "terrains", "atelier", "ateliers",
}

# Recognize French prices such as 850 €, 1 200 €, 1.035 € and 1 250,50 €.
EURO_PATTERN = r"(?<!\d)(\d{1,3}(?:[.\s\u00a0\u202f]\d{3})+|\d+)(?:,(\d{1,2}))?\s*€"


def euro_value(match):
    """Convert a matched French price to a number, leaving missing prices empty."""
    if match is None:
        return None
    whole = re.sub(r"[.\s\u00a0\u202f]", "", match.group(1))
    decimals = match.group(2)
    return float(whole + ("." + decimals if decimals else ""))


def is_access_check(page):
    """Recognize PAP access-check pages in French and English."""
    title = page.title().casefold()
    return any(word in title for word in (
        "un instant", "just a moment", "cloudflare", "access denied"
    ))


def wait_for_access(page, selector, timeout_ms=60000):
    """Wait if an access check appears; never parse it as a real advertisement.

    Chromium remains visible for permitted human interaction. A website can
    still refuse automated access even after the waiting period.
    """
    if not is_access_check(page):
        return True

    print("  Website access check displayed in Chromium.", flush=True)
    try:
        page.wait_for_selector(selector, timeout=timeout_ms)
    except TargetClosedError:
        print("  Chromium was closed during the access check.", flush=True)
        return False
    except PlaywrightTimeoutError:
        print("  The website did not show the expected content in time.", flush=True)
        return False

    if is_access_check(page):
        print("  The website is still displaying an access check.", flush=True)
        return False
    return True


def property_type_from_title(soup):
    """Read the listing's <h1> and keep only recognizable residential types.

    'Bureaux à Marseille' is rejected. 'Appartement avec bureau' stays an
    apartment, because the property category is 'appartement'.
    """
    title_tag = soup.find("h1")
    title = title_tag.get_text(" ", strip=True) if title_tag else ""
    words = re.findall(r"[a-zà-ÿ]+", title.casefold())

    # Explicitly commercial titles should never enter a housing dataset.
    if "bureaux" in words or "commerces" in words:
        return title, None
    if "commercial" in words and ("local" in words or "locaux" in words):
        return title, None

    # The first stated property category determines the listing type.
    for word in words:
        if word in NON_RESIDENTIAL_TYPES:
            return title, None
        if word in RESIDENTIAL_TYPES:
            return title, RESIDENTIAL_TYPES[word]

    # Without a clear residential category, we do not guess.
    return title, None


def collect_listing_urls(page, max_pages):
    """Collect unique individual ad links from PAP search result cards."""
    all_urls = set()
    print("Collecting PAP advertisement links...", flush=True)

    for page_num in range(1, max_pages + 1):
        url = BASE_URL if page_num == 1 else f"{BASE_URL}-{page_num}"
        print(f"Search page {page_num}/{max_pages}", flush=True)

        try:
            page.goto(url, wait_until="domcontentloaded", timeout=60000)
            time.sleep(2)  # Allow the result page to finish loading.

            # Accept the cookie banner if it is displayed.
            try:
                page.locator("#didomi-notice-agree-button").click(timeout=1000)
            except PlaywrightTimeoutError:
                pass

            if not wait_for_access(page, ".search-list-item"):
                # Keep URLs collected on previous pages if access stops here.
                print("  Keeping the links collected so far.", flush=True)
                break

            html = page.content()
        except TargetClosedError:
            print("  Chromium was closed. Stopping URL collection.", flush=True)
            break
        except Exception as error:
            print(f"  Search page could not be loaded: {error}", flush=True)
            continue

        # Keep the card-based extraction used in the working script.
        soup = BeautifulSoup(html, "html.parser")
        ad_containers = soup.find_all(
            "div", class_=re.compile(r"search-list-item|annonce-item")
        )

        if not ad_containers:
            print("  No result cards found on this page.", flush=True)
            break

        new_urls = 0
        for ad in ad_containers:
            link_tag = ad.find("a", href=True)
            if link_tag is None:
                continue

            href = link_tag["href"]
            if "/annonces/" not in href:
                continue

            full_url = urljoin(PAP_DOMAIN, href)
            if full_url not in all_urls:
                all_urls.add(full_url)
                new_urls += 1

        print(f"  Found {new_urls} new links.", flush=True)
        time.sleep(1)

    print(f"Collected {len(all_urls)} unique advertisement links.", flush=True)
    return sorted(all_urls)
    
def pap_furnished(text):
    if re.search(r"\b(non[- ]meubl[ée]e?|pas meubl[ée]e?|sans meubles?|location nue)\b", text, re.I):
        return "Non"

    if re.search(r"\b(appartement|studio|logement|location|maison)\s+meubl[ée]e?\b", text, re.I):
        return "Oui"

    return None

def extract_listing(soup, ad_url, property_type, title):
    """Extract values from one residential listing and return one dictionary."""
    page_text = soup.get_text(separator=" ", strip=True)

    # Price: match French euro formats (including dots and spaces in thousands).
    price_match = re.search(EURO_PATTERN, page_text)
    price = euro_value(price_match)

    surface_match = re.search(
        r"(\d+(?:[.,]\d+)?)\s*(?:m²|m2)", page_text, re.IGNORECASE
    )
    rooms_match = re.search(r"(\d+)\s*pièce", page_text, re.IGNORECASE)
    bedrooms_match = re.search(r"(\d+)\s*chambre", page_text, re.IGNORECASE)
    zip_match = re.search(r"\b(130\d{2})\b", page_text)

    # Read fees and deposits near their own labels, not as the first price.
    charges_match = re.search(
        r"charges.{0,80}?" + EURO_PATTERN, page_text, re.IGNORECASE
    )
    deposit_match = re.search(
        r"garantie.{0,80}?" + EURO_PATTERN, page_text, re.IGNORECASE
    )
    dpe_match = re.search(r"DPE\s*:\s*([A-G])", page_text, re.IGNORECASE)
    ges_match = re.search(r"GES\s*:\s*([A-G])", page_text, re.IGNORECASE)

    return {
        "source_site": "PAP",
        "url": ad_url,
        "title": title,
        "property_type": property_type,
        "price_total_eur": price,
        "surface_m2": (
            float(surface_match.group(1).replace(",", "."))
            if surface_match else None
        ),
        "rooms": int(rooms_match.group(1)) if rooms_match else None,
        "bedrooms": int(bedrooms_match.group(1)) if bedrooms_match else None,
        "zipcode": zip_match.group(1) if zip_match else None,
        "charges_eur": euro_value(charges_match),
        "depot_garantie_eur": euro_value(deposit_match),
        "dpe_class": dpe_match.group(1).upper() if dpe_match else None,
        "ges_class": ges_match.group(1).upper() if ges_match else None,
        "is_furnished": pap_furnished(page_text),
        "has_elevator": "Oui" if re.search(r"\bascenseur\b", page_text, re.IGNORECASE) else "Non",
        "has_balcony_terrace": "Oui" if re.search(r"\b(balcon|terrasse)\b", page_text, re.IGNORECASE) else "Non",
        "raw_text": page_text[:2000],
        "honoraires_eur": 0,  # Project assumption: PAP rentals are person-to-person.
    }


def scrape_pap_deep_data(max_pages=20, max_ads=None):
    """Open PAP in Chromium, collect links and extract residential listings."""
    print("Launching Chromium browser...", flush=True)
    all_records = []

    with sync_playwright() as p:
        browser = p.chromium.launch(
            headless=False,
            args=["--disable-blink-features=AutomationControlled"],
        )
        try:
            context = browser.new_context(
                user_agent=(
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36"
                ),
                viewport={"width": 1280, "height": 800},
            )
            page = context.new_page()
            urls = collect_listing_urls(page, max_pages)

            print("Opening individual PAP advertisements...", flush=True)
            for index, ad_url in enumerate(urls, start=1):
                if max_ads is not None and len(all_records) >= max_ads:
                    break

                print(f"Listing {index}/{len(urls)}: {ad_url}", flush=True)
                try:
                    page.goto(
                        ad_url, wait_until="domcontentloaded", timeout=30000
                    )
                    time.sleep(1.5)  # Allow the page to load fully.

                    if not wait_for_access(page, ".item-title"):
                        print("  Skipping a page that did not become accessible.", flush=True)
                        continue

                    soup = BeautifulSoup(page.content(), "html.parser")
                except TargetClosedError:
                    print("  Chromium was closed. Keeping already extracted data.", flush=True)
                    break
                except Exception as error:
                    print(f"  Could not read this listing: {error}", flush=True)
                    continue

                # Decide whether this is housing BEFORE extracting any fields.
                title, property_type = property_type_from_title(soup)
                if property_type is None:
                    print(f"  Skipping non-residential/unknown title: {title!r}", flush=True)
                    continue

                record = extract_listing(soup, ad_url, property_type, title)
                all_records.append(record)
                print(f"  Saved residential listing #{len(all_records)}.", flush=True)

        finally:
            # Close the browser even if a page cannot be loaded.
            try:
                browser.close()
            except TargetClosedError:
                pass

    df = pd.DataFrame(all_records)
    if not df.empty:
        # A surface of zero cannot be used to calculate rent per square metre.
        valid = df["price_total_eur"].notna() & df["surface_m2"].gt(0)
        df.loc[valid, "price_per_m2"] = (
            df.loc[valid, "price_total_eur"] / df.loc[valid, "surface_m2"]
        ).round(2)

    return df


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Scrape residential PAP rentals")
    parser.add_argument("--max-pages", type=int, default=2)
    parser.add_argument("--max-ads", type=int, default=10)
    parser.add_argument("--output", default="pap_marseille_deep_data.csv")
    args = parser.parse_args()

    limit = args.max_ads if args.max_ads > 0 else None
    df_pap = scrape_pap_deep_data(max_pages=args.max_pages, max_ads=limit)

    if df_pap.empty:
        print("No residential advertisements were extracted.", flush=True)
        raise SystemExit(1)

    output_file = Path(args.output).resolve()
    output_file.parent.mkdir(parents=True, exist_ok=True)
    df_pap.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df_pap)} PAP residential listings to {output_file}", flush=True)


Overwriting pap_worker.py


In [8]:
RUN_PAP = True  # Set to False when you do not want to open Chromium.
PAP_MAX_PAGES = 20
PAP_MAX_ADS = 0    # Count residential listings only. Use 0 for no limit.

# The previous %%writefile cell creates the worker in this notebook's folder.
worker_file = Path("pap_worker.py").resolve()
pap_output = (OUTPUT_DIR / "pap.csv").resolve()

if RUN_PAP:
    if not worker_file.exists():
        raise FileNotFoundError("Run the %%writefile pap_worker.py cell first.")

    command = [
        sys.executable, "-u", str(worker_file),
        "--max-pages", str(PAP_MAX_PAGES),
        "--max-ads", str(PAP_MAX_ADS),
        "--output", str(pap_output),
    ]
    print("Starting the PAP worker in a separate Python process...", flush=True)

    with subprocess.Popen(
        command,
        cwd=str(worker_file.parent),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        exit_code = process.wait()

    print("PAP worker exit code:", exit_code)
    if exit_code == 0 and pap_output.exists():
        df_pap = pd.read_csv(pap_output, dtype={"zipcode": "string"})
        print("Residential PAP listings:", len(df_pap))
        display(df_pap.head())
    else:
        print("No new PAP dataset was saved. See the messages above.")
else:
    print("PAP collection is disabled.")


Starting the PAP worker in a separate Python process...
Launching Chromium browser...
Search page 1/20
  Found 13 new links.
Search page 2/20
  Found 14 new links.
Search page 3/20
  Found 13 new links.
Search page 4/20
  Found 12 new links.
Search page 5/20
  Found 6 new links.
Search page 6/20
  Found 11 new links.
Search page 7/20
  Found 12 new links.
Search page 8/20
  Found 12 new links.
Search page 9/20
  Found 10 new links.
Search page 10/20
  Found 0 new links.
Search page 11/20
  Found 0 new links.
Search page 12/20
  Found 0 new links.
Search page 13/20
  Found 0 new links.
Search page 14/20
  Found 0 new links.
Search page 15/20
  Found 0 new links.
Search page 16/20
  Found 0 new links.
Search page 17/20
  Found 0 new links.
Search page 18/20
  Found 0 new links.
Search page 19/20
  Found 0 new links.
Search page 20/20
  Found 0 new links.
Collected 103 unique advertisement links.
Opening individual PAP advertisements...
Listing 1/103: https://www.pap.fr/annonces/apparteme

,source_site,url,title,property_type,price_total_eur,surface_m2,rooms,bedrooms,zipcode,charges_eur,depot_garantie_eur,dpe_class,ges_class,is_furnished,has_elevator,has_balcony_terrace,raw_text,honoraires_eur,price_per_m2
0,PAP,https://www.pap.fr/annonces/appartement-marsei...,Location meublée appartement 2 pièces 40 m² Ma...,appartement,861.0,40.00,2,1.0,13010,861.0,0.0,NaN,NaN,Oui,Non,Oui,Location meublée appartement 2 pièces 40 m² Ma...,0,21.52
1,PAP,https://www.pap.fr/annonces/appartement-marsei...,Location meublée appartement 3 pièces 70 m² Ma...,appartement,1350.0,70.00,3,2.0,13010,2000.0,2000.0,NaN,NaN,Oui,Non,Oui,Location meublée appartement 3 pièces 70 m² Ma...,0,19.29
2,PAP,https://www.pap.fr/annonces/appartement-marsei...,Location appartement 3 pièces 60 m² Marseille ...,appartement,1014.0,60.00,3,2.0,13010,1014.0,934.0,NaN,NaN,Non,Oui,Oui,Location appartement 3 pièces 60 m² Marseille ...,0,16.90
3,PAP,https://www.pap.fr/annonces/appartement-marsei...,Location meublée studio 24 m² Marseille 10E (1...,studio,670.0,24.00,1,NaN,13010,630.0,1260.0,NaN,NaN,Oui,Oui,Non,Location meublée studio 24 m² Marseille 10E (1...,0,27.92
4,PAP,https://www.pap.fr/annonces/appartement-marsei...,"Location meublée appartement 3 pièces 58,27 m²...",appartement,1100.0,58.27,3,2.0,13011,100.0,2000.0,D,NaN,Oui,Oui,Non,"Location meublée appartement 3 pièces 58,27 m²...",0,18.88


<h1 style="color: #17638c; font-size: 24px; font-weight: 750; margin: 2px 0 16px 0; line-height: 1.35;">4. Review the extracted files (no additional scraping)</h1>


In [9]:
files = {
    'Bienici': OUTPUT_DIR / 'bienici.csv',
    'PAP': OUTPUT_DIR / 'pap.csv',
    'ParuVendu': OUTPUT_DIR / 'paruvendu.csv',
}

for source, filename in files.items():
    print(f'\n--- {source} ---')
    if not filename.exists():
        print('Not collected in this run:', filename)
        continue
    df = pd.read_csv(filename, dtype={'zipcode': 'string'})
    print('Rows:', len(df), '| Columns:', len(df.columns))
    print('Missing values in key columns:')
    keys = [col for col in ('price_total_eur', 'surface_m2', 'zipcode', 'dpe_class', 'ges_class') if col in df.columns]
    display(df[keys].isna().sum().to_frame('missing_rows'))
    display(df.head(3))


--- Bienici ---
Rows: 685 | Columns: 28
Missing values in key columns:


,missing_rows
price_total_eur,1
surface_m2,2
zipcode,0
dpe_class,0
ges_class,11


,source_site,id,url,title,property_type,price_total_eur,charges_eur,depot_garantie_eur,honoraires_eur,surface_m2,...,longitude,dpe_class,ges_class,heating,is_furnished,has_elevator,has_balcony_terrace,has_cellar,has_parking,raw_text
0,Bienici,ag132391-555657911,NaN,Atelier / Local / Stockage de 17m2 Quartier Ti...,loft,285.0,35.0,NaN,350.00,17.00,...,5.397644,NS,NS,NaN,NaN,NaN,NaN,NaN,NaN,Atelier / Local / Stockage de 17m2 Quartier Ti...
1,Bienici,hektor-xavierradisson13-203,NaN,Studio 26 rue Aramis 13014,flat,359.0,69.0,290.0,187.00,14.26,...,5.386046,C,B,électrique individuel,Non,NaN,NaN,NaN,NaN,Contact : 06 80 12 05 66 . Situé 26 rue Aramis...
2,Bienici,seiitra-elyade_bi-17290,NaN,NaN,flat,392.0,65.0,327.0,246.91,18.82,...,5.432999,A,A,Convecteurs Electrique Individuel,Non,Oui,NaN,NaN,NaN,Résidence étudiante idéalement située à Marsei...



--- PAP ---
Rows: 60 | Columns: 19
Missing values in key columns:


,missing_rows
price_total_eur,0
surface_m2,0
zipcode,0
dpe_class,56
ges_class,60


,source_site,url,title,property_type,price_total_eur,surface_m2,rooms,bedrooms,zipcode,charges_eur,depot_garantie_eur,dpe_class,ges_class,is_furnished,has_elevator,has_balcony_terrace,raw_text,honoraires_eur,price_per_m2
0,PAP,https://www.pap.fr/annonces/appartement-marsei...,Location meublée appartement 2 pièces 40 m² Ma...,appartement,861.0,40.0,2,1.0,13010,861.0,0.0,NaN,NaN,Oui,Non,Oui,Location meublée appartement 2 pièces 40 m² Ma...,0,21.52
1,PAP,https://www.pap.fr/annonces/appartement-marsei...,Location meublée appartement 3 pièces 70 m² Ma...,appartement,1350.0,70.0,3,2.0,13010,2000.0,2000.0,NaN,NaN,Oui,Non,Oui,Location meublée appartement 3 pièces 70 m² Ma...,0,19.29
2,PAP,https://www.pap.fr/annonces/appartement-marsei...,Location appartement 3 pièces 60 m² Marseille ...,appartement,1014.0,60.0,3,2.0,13010,1014.0,934.0,NaN,NaN,Non,Oui,Oui,Location appartement 3 pièces 60 m² Marseille ...,0,16.90



--- ParuVendu ---
Rows: 300 | Columns: 22
Missing values in key columns:


,missing_rows
price_total_eur,1
surface_m2,1
zipcode,14
dpe_class,100
ges_class,300


,price_total_eur,charges_eur,depot_garantie_eur,honoraires_eur,surface_m2,rooms,bedrooms,zipcode,dpe_class,ges_class,...,is_furnished,has_elevator,has_balcony_terrace,heating,floor,exposure,source_site,url,property_type,raw_text
0,1150.0,NaN,NaN,NaN,75.0,3,NaN,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ParuVendu,https://www.paruvendu.fr/immobilier/location/a...,appartement,Annonce tel beau t3 impeccable quartier calme ...
1,1490.0,NaN,NaN,NaN,40.0,2,NaN,13006,NaN,NaN,...,Oui,NaN,NaN,NaN,NaN,NaN,ParuVendu,https://www.paruvendu.fr/immobilier/location/a...,appartement,Annonce location appartement 2 pièces de 40m2 ...
2,555.0,NaN,NaN,NaN,35.0,2,NaN,13014,E,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ParuVendu,https://www.paruvendu.fr/immobilier/location/a...,appartement,Annonce location appartement 2 pièces de 35m2 ...
